In [1]:
# BODAQS combined auto-preprocess + simple suspension metrics workflow
#
# Edit the user parameters below, then run this cell.

import hashlib
import html
import json
from pathlib import Path

import ipywidgets as W
import pandas as pd
from IPython.display import display

from bodaqs_analysis import run_macro
from bodaqs_analysis.artifacts import (
    ArtifactStore,
    copy_raw_csv_to_source,
    copy_session_aux_sources,
    ensure_run_is_new,
    ensure_session_is_new,
    make_run_id,
    save_session_artifacts,
    set_run_description,
    set_session_description,
    write_events_partitioned_by_schema_id,
    write_metrics_partitioned_by_schema_id,
    write_run_manifest,
    write_session_manifest,
)
from bodaqs_analysis.dashboards import make_simple_suspension_metrics_dashboard
from bodaqs_analysis.ui.preprocess_file_selector import load_processed_sha256_set
from bodaqs_analysis.widgets.session_selector import make_session_selector

# --- User parameters -------------------------------------------------------
LOG_DIR = Path(r"C:\Users\Ben Connor\OneDrive\Documents\logs\Test logs")
PROFILE_PATH = Path("config/preprocess_profiles/suspension_default_v1.json")
ARTIFACTS_DIR = Path("artifacts")
FIT_DIR = Path(r"C:\Users\Ben Connor\OneDrive\Documents\logs\GPS")
FIT_BINDINGS_PATH = Path("config/fit_bindings_v1.json")
PROMPT_FOR_DESCRIPTIONS = False

# Optional tuning parameters for file discovery / run metadata
CSV_GLOB = "*.CSV"
INCLUDE_LOWERCASE_CSV = True
RUN_TZ_LABEL = "AWST"
SHA_CACHE_FILE = Path(".bodaqs_preprocess_sha_cache.json")


NOTEBOOK_ROOT = Path.cwd()
PROFILE_SCHEMA = "bodaqs.preprocess_profile"
PROFILE_VERSION = 1


def _resolve_from_cwd(path_like):
    p = Path(path_like).expanduser()
    return p.resolve() if p.is_absolute() else (NOTEBOOK_ROOT / p).resolve()


def _resolve_optional_from_cwd(path_like):
    if path_like is None:
        return None
    return _resolve_from_cwd(path_like)


def _read_json(path: Path, default):
    try:
        if path.exists():
            return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        pass
    return default


def _write_json(path: Path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, indent=2, sort_keys=True), encoding="utf-8")


def _sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


def _cache_key(path: Path) -> str:
    try:
        st = path.stat()
        return f"{str(path.resolve())}|{st.st_size}|{int(st.st_mtime)}"
    except Exception:
        return str(path.resolve())


def _iter_csv_files(log_dir: Path, file_glob: str, include_lowercase_csv: bool):
    files = []
    files.extend(sorted(log_dir.glob(file_glob)))
    if include_lowercase_csv and file_glob.upper() == "*.CSV":
        files.extend(sorted(log_dir.glob("*.csv")))
    return sorted({p.resolve() for p in files if p.is_file()})


def _scan_unprocessed_csv_files(
    *,
    log_dir: Path,
    artifacts_dir: Path,
    sha_cache_file: Path,
    file_glob: str,
    include_lowercase_csv: bool,
):
    processed_sha = load_processed_sha256_set(artifacts_dir)
    sha_cache = _read_json(sha_cache_file, {})

    if not log_dir.exists() or not log_dir.is_dir():
        return {
            "files": [],
            "new_files": [],
            "processed_files": [],
            "hash_misses": 0,
            "processed_sha_count": len(processed_sha),
            "warning": f"Log directory does not exist or is not a directory: {log_dir}",
        }

    files = _iter_csv_files(log_dir, file_glob=file_glob, include_lowercase_csv=include_lowercase_csv)
    new_files = []
    processed_files = []
    hash_misses = 0

    for path in files:
        key = _cache_key(path)
        rec = sha_cache.get(key)
        sha = None
        if isinstance(rec, dict):
            value = rec.get("sha256")
            if isinstance(value, str) and value:
                sha = value
        if sha is None:
            sha = _sha256_file(path)
            sha_cache[key] = {"sha256": sha}
            hash_misses += 1

        if sha in processed_sha:
            processed_files.append(path)
        else:
            new_files.append(path)

    if hash_misses:
        _write_json(sha_cache_file, sha_cache)

    return {
        "files": files,
        "new_files": new_files,
        "processed_files": processed_files,
        "hash_misses": hash_misses,
        "processed_sha_count": len(processed_sha),
        "warning": None,
    }


def _load_preprocess_profile(profile_path: Path):
    if not profile_path.exists():
        raise FileNotFoundError(f"Preprocess profile not found: {profile_path}")

    obj = json.loads(profile_path.read_text(encoding="utf-8"))
    if obj.get("schema") != PROFILE_SCHEMA:
        raise ValueError(
            f"Unexpected preprocess profile schema: {obj.get('schema')!r} (expected {PROFILE_SCHEMA!r})"
        )
    if int(obj.get("version", -1)) != PROFILE_VERSION:
        raise ValueError(
            f"Unexpected preprocess profile version: {obj.get('version')!r} (expected {PROFILE_VERSION})"
        )

    cfg = obj.get("config")
    if not isinstance(cfg, dict):
        raise ValueError("Preprocess profile is missing a 'config' object")

    required_keys = {
        "schema_path",
        "strict",
        "zeroing_enabled",
        "zero_window_s",
        "zero_min_samples",
        "clip_0_1",
        "butterworth_smoothing",
        "butterworth_generate_residuals",
        "active_signal_disp_col",
        "active_disp_thresh",
        "active_vel_thresh",
        "active_window",
        "active_padding",
        "active_min_seg",
        "normalize_ranges",
    }
    missing = sorted(required_keys - set(cfg.keys()))
    if missing:
        raise ValueError("Preprocess profile is missing required config keys: " + ", ".join(missing))

    normalize_ranges = cfg.get("normalize_ranges")
    if not isinstance(normalize_ranges, dict) or not normalize_ranges:
        raise ValueError("Preprocess profile 'normalize_ranges' must be a non-empty object")

    smoothing = cfg.get("butterworth_smoothing")
    if not isinstance(smoothing, list):
        raise ValueError("Preprocess profile 'butterworth_smoothing' must be a list")

    return obj, dict(cfg)


def _html_box(title: str, lines, *, background: str, border: str):
    items_html = "".join(f"<li>{html.escape(str(line))}</li>" for line in lines)
    return W.HTML(
        value=(
            f"<div style='border:1px solid {border}; background:{background}; padding:12px; margin:6px 0;'>"
            f"<div style='font-weight:700; margin-bottom:8px;'>{html.escape(title)}</div>"
            f"<ul style='margin:0 0 0 18px; padding:0;'>{items_html}</ul>"
            "</div>"
        )
    )


log_dir = _resolve_from_cwd(LOG_DIR)
profile_path = _resolve_from_cwd(PROFILE_PATH)
artifacts_dir = _resolve_from_cwd(ARTIFACTS_DIR)
fit_dir = _resolve_optional_from_cwd(FIT_DIR)
fit_bindings_path = _resolve_optional_from_cwd(FIT_BINDINGS_PATH)
sha_cache_file = _resolve_from_cwd(SHA_CACHE_FILE)
store = ArtifactStore(artifacts_dir)

scan = _scan_unprocessed_csv_files(
    log_dir=log_dir,
    artifacts_dir=artifacts_dir,
    sha_cache_file=sha_cache_file,
    file_glob=CSV_GLOB,
    include_lowercase_csv=INCLUDE_LOWERCASE_CSV,
)

status_lines = [
    f"Notebook root: {NOTEBOOK_ROOT}",
    f"Artifact root: {artifacts_dir}",
    f"Log directory: {log_dir}",
    f"FIT directory: {fit_dir}",
    f"FIT bindings path: {fit_bindings_path}",
    f"CSV files found: {len(scan['files'])}",
    f"Already processed (by source SHA): {len(scan['processed_files'])}",
    f"New files to process: {len(scan['new_files'])}",
    f"Processed SHA entries discovered: {scan['processed_sha_count']}",
]

warning_lines = []
if scan["hash_misses"]:
    status_lines.append(f"SHA cache misses this run: {scan['hash_misses']}")
if scan["warning"]:
    warning_lines.append(scan["warning"])

run_id = None
processed_session_ids = []
profile = None

if scan["new_files"]:
    profile, cfg = _load_preprocess_profile(profile_path)
    fit_cfg = dict(cfg.get("fit_import") or {})
    if fit_cfg:
        if fit_dir is not None:
            fit_cfg["fit_dir"] = str(fit_dir)
        elif fit_cfg.get("fit_dir"):
            fit_cfg["fit_dir"] = str(_resolve_from_cwd(fit_cfg["fit_dir"]))
        if fit_bindings_path is not None:
            fit_cfg["bindings_path"] = str(fit_bindings_path)
        elif fit_cfg.get("bindings_path"):
            fit_cfg["bindings_path"] = str(_resolve_from_cwd(fit_cfg["bindings_path"]))
        cfg["fit_import"] = fit_cfg
        status_lines.append(f"FIT import enabled: {bool(fit_cfg.get('enabled'))}")
        status_lines.append(f"Resolved FIT directory: {fit_cfg.get('fit_dir')}")
    schema_path = _resolve_from_cwd(cfg["schema_path"])
    if not schema_path.exists():
        raise FileNotFoundError(f"Schema path from profile does not exist: {schema_path}")

    run_id_base = make_run_id(tz_label=RUN_TZ_LABEL)
    run_id = run_id_base
    suffix = 1
    while store.run_dir(run_id).exists():
        run_id = f"{run_id_base}_{suffix:02d}"
        suffix += 1
    ensure_run_is_new(store, run_id=run_id, force=False)

    events_all = []
    metrics_all = []

    print(f"Processing {len(scan['new_files'])} new file(s) into run {run_id} ...")
    for csv_path in scan["new_files"]:
        print(f"  - {csv_path.name}")
        results = run_macro(
            str(csv_path),
            str(schema_path),
            zeroing_enabled=bool(cfg["zeroing_enabled"]),
            zero_window_s=float(cfg["zero_window_s"]),
            zero_min_samples=int(cfg["zero_min_samples"]),
            clip_0_1=bool(cfg["clip_0_1"]),
            active_signal_disp_col=cfg.get("active_signal_disp_col"),
            active_signal_vel_col=cfg.get("active_signal_vel_col"),
            active_disp_thresh=float(cfg["active_disp_thresh"]),
            active_vel_thresh=float(cfg["active_vel_thresh"]),
            active_window=str(cfg["active_window"]),
            active_padding=str(cfg["active_padding"]),
            active_min_seg=str(cfg["active_min_seg"]),
            normalize_ranges=dict(cfg["normalize_ranges"]),
            fit_import=cfg.get("fit_import"),
            sample_rate_hz=cfg.get("sample_rate_hz"),
            butterworth_smoothing=list(cfg.get("butterworth_smoothing", [])),
            butterworth_generate_residuals=bool(cfg["butterworth_generate_residuals"]),
            strict=bool(cfg["strict"]),
        )

        session = results["session"]
        session_id = str(session["session_id"])
        ensure_session_is_new(store, run_id=run_id, session_id=session_id, force=False)
        processed_session_ids.append(session_id)

        source_sha256 = copy_raw_csv_to_source(
            store=store,
            run_id=run_id,
            session_id=session_id,
            csv_path=csv_path,
        )
        aux_sources = copy_session_aux_sources(
            store=store,
            run_id=run_id,
            session_id=session_id,
            aux_sources=session.get("source", {}).get("aux_sources"),
        )
        save_session_artifacts(
            store,
            run_id=run_id,
            session_id=session_id,
            session_df=session["df"],
            session_meta=session["meta"],
            secondary_stream_dfs=session.get("stream_dfs"),
            secondary_stream_meta=session.get("meta", {}).get("secondary_streams"),
        )

        events_df = results.get("events", pd.DataFrame())
        metrics_df = results.get("metrics", pd.DataFrame())
        write_session_manifest(
            store,
            run_id=run_id,
            session_id=session_id,
            contracts={"session": "v0.x", "events": "v0.x", "metrics": "v0.x"},
            source={"path": "source/input.csv", "sha256": source_sha256},
            aux_sources=aux_sources,
            summary={
                "n_rows": int(len(session["df"])),
                "n_events": int(len(events_df)) if isinstance(events_df, pd.DataFrame) else 0,
                "n_metrics": int(len(metrics_df)) if isinstance(metrics_df, pd.DataFrame) else 0,
            },
        )

        if isinstance(events_df, pd.DataFrame) and not events_df.empty:
            events_all.append(events_df)
            write_events_partitioned_by_schema_id(
                store=store,
                run_id=run_id,
                session_id=session_id,
                events_df=events_df,
                schema_path=schema_path,
            )

        if isinstance(metrics_df, pd.DataFrame) and not metrics_df.empty:
            metrics_all.append(metrics_df)
            write_metrics_partitioned_by_schema_id(
                store=store,
                run_id=run_id,
                session_id=session_id,
                metrics_df=metrics_df,
            )

    write_run_manifest(
        store,
        run_id=run_id,
        session_ids=processed_session_ids,
        timezone_label=RUN_TZ_LABEL,
        pipeline_config={
            "profile_schema": profile.get("schema"),
            "profile_version": profile.get("version"),
            "profile_id": profile.get("profile_id"),
            "profile_path": str(profile_path),
            "schema_path": str(schema_path),
            "zeroing_enabled": bool(cfg["zeroing_enabled"]),
            "normalize_ranges": bool(cfg["normalize_ranges"]),
            "butterworth_smoothing": bool(cfg.get("butterworth_smoothing")),
            "butterworth_generate_residuals": bool(cfg["butterworth_generate_residuals"]),
            "fit_import_enabled": bool((cfg.get("fit_import") or {}).get("enabled")),
            "fit_dir": (cfg.get("fit_import") or {}).get("fit_dir"),
            "fit_bindings_path": (cfg.get("fit_import") or {}).get("bindings_path"),
            "sample_rate_hz": cfg.get("sample_rate_hz"),
            "ingestion_mode": "strict" if cfg["strict"] else "tolerant",
            "n_files": int(len(scan["new_files"])),
        },
    )

    status_lines.append(f"New run written: {run_id}")
    status_lines.append(f"Processed session ids: {', '.join(processed_session_ids)}")

    if PROMPT_FOR_DESCRIPTIONS:
        run_desc = input(f"Run description for {run_id} (blank to skip): ").strip()
        set_run_description(store, run_id=run_id, description=run_desc)
        for session_id in processed_session_ids:
            session_desc = input(
                f"Session {session_id} description (blank to skip, '.' to stop): "
            ).strip()
            if session_desc == ".":
                break
            set_session_description(
                store,
                run_id=run_id,
                session_id=session_id,
                description=session_desc,
            )
        status_lines.append("Run/session description prompts completed.")
else:
    status_lines.append("No new CSV files were found; proceeding directly to existing artifacts.")

summary_box = _html_box(
    "Run Summary",
    status_lines,
    background="#f8fafc",
    border="#d1d5db",
)

ui_children = [summary_box]
if warning_lines:
    ui_children.append(
        _html_box(
            "Warnings",
            warning_lines,
            background="#fff7ed",
            border="#fdba74",
        )
    )

selector = make_session_selector(artifacts_dir=artifacts_dir, select_first_by_default=True)
dashboard = make_simple_suspension_metrics_dashboard(selector)
ui_children.extend([selector["ui"], dashboard["ui"]])

limitations_lines = [
    "Unprocessed files are detected by raw CSV SHA only. If the schema or preprocessing profile changes later, previously processed logs will still be skipped unless you remove or rework the old artifacts.",
    "The preprocessing profile is currently a practical JSON artifact rather than a formally documented contract. A contract document for it still needs to be written.",
    "A single profile is applied to every new log found in LOG_DIR, so this notebook assumes those logs share compatible signal naming and normalization ranges.",
    "After preprocessing, the standard selector and dashboard are shown across the normal artifact library. They are not automatically restricted to only the newly processed batch.",
    "If there are no artifacts yet and no new logs are processed, the selector and dashboard may appear empty until a first successful preprocessing run is completed.",
]
ui_children.append(
    _html_box(
        "Current limitations",
        limitations_lines,
        background="#fefce8",
        border="#fde047",
    )
)

display(W.VBox(ui_children, layout=W.Layout(width="100%", gap="12px")))


Processing 2 new file(s) into run run_2026-04-22T10-06-38_AWST ...
  - 2026-02-01_10-50-01.CSV
  - 2026-02-20_13-08-45.CSV
